# Reasoning Models — OpenAI-Compatible Endpoints

Tests reasoning/thinking models (o1, QwQ, etc.) via Databricks serving endpoints using the OpenAI SDK. Includes logic puzzle examples to exercise multi-step reasoning.

In [0]:
puzzle = """
Four hikers (Ava, Ben, Ilya, Noor) each returned at a different time (6:10, 6:20, 6:30, 6:40 PM) to a different cabin (Cedar, Birch, Spruce, Aspen) carrying a different item (Compass, Journal, Lantern, Map). Determine who returned when, to which cabin, and with what.
Clues:
1) The person who returned at 6:20 went to Cedar, but did not carry the Lantern.
2) Noor didn’t go to Aspen and didn’t carry the Map.
3) Ava returned 10 minutes before the person who went to Birch.
4) The Compass carrier returned later than Ilya but earlier than the person headed to Spruce.
5) Ben went to Birch.
6) The Journal wasn’t taken to Cedar.
7) The person who went to Spruce returned 10 minutes after the Lantern carrier.
8) Ilya returned at 6:10 or 6:20.

Expected answer format:
6:10 — Person — Cabin — Item
6:20 — Person — Cabin — Item
6:30 — Person — Cabin — Item
6:40 — Person — Cabin — Item
"""

In [0]:
"""
Answer key:
6:10 — Ilya — Aspen — Map
6:20 — Ava — Cedar — Compass
6:30 — Ben — Birch — Lantern
6:40 — Noor — Spruce — Journal
"""

In [0]:
from databricks_langchain import ChatDatabricks

llm = ChatDatabricks(
    model="databricks-gpt-oss-120b",
    extra_params={
        "reasoning_effort": "low"
    }
    )

response = llm.invoke(puzzle)

In [0]:
from pprint import pprint
pprint(response.content[0]['summary'][0]['text'])

In [0]:
from openai import OpenAI

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
workspace_url = w.config.host
token = w.config.token

client = OpenAI(
    base_url=f"{workspace_url}/serving-endpoints",
    api_key=token
)

response = client.chat.completions.create(
    model="databricks-gpt-oss-120b",
    messages=[
        {"role": "user", "content": puzzle}
    ],
    reasoning_effort="low"
    )

In [0]:
pprint(response.choices[0].message.content[0]['summary'][0]['text'])